# 🔧 Feature Engineering

**Features añadidos (solo los más importantes):**
1. `insider_id`: Identificador único del insider (ticker + role)
2. `insider_trade_count`: Track record del insider (experiencia)
3. `days_since_last_trade`: Frecuencia de trading
4. `trade_day_of_week`: Timing de la operación
5. `cluster_c_level_pct`: Calidad del cluster (% de C-levels)

**Decisión de diseño:** Solo incluimos features con alto potencial predictivo para evitar ruido en el modelo.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

## 1. Cargar Datos

In [2]:
project_root = Path("../..")
input_path = project_root / "data" / "processed" / "insider_trades_with_targets.parquet"

if not input_path.exists():
    print(f"❌ Error: {input_path} no encontrado")
    print("👉 Primero ejecuta el notebook 02_tiingo_integration con TODOS los datos")
    raise FileNotFoundError(f"File not found: {input_path}")

df = pd.read_parquet(input_path)

print(f"✅ Datos cargados: {len(df):,} transacciones")
print(f"   Tickers únicos: {df['ticker'].nunique():,}")
print(f"   Rango: {df['trade_date'].min()} → {df['trade_date'].max()}")
df.head()

✅ Datos cargados: 45,242 transacciones
   Tickers únicos: 5,512
   Rango: 2017-10-10 00:00:00 → 2026-01-29 00:00:00


,ticker,filing_date,trade_date,insider_role,transaction_value,delta_owned,owned_pct_change,reporting_lag,is_c_level,cluster_buy,cluster_size,return_1w,return_1m,return_3m
0,ICTSF,2018-04-10 17:17:23,2018-04-04,Dir,132000.0,0.494383,98.0,6.0,False,5.0,5.0,0.016667,0.083333,0.200000
1,AVGO,2018-04-17 18:18:09,2018-04-13,Dir,85478486.0,0.168234,20.0,4.0,False,1.0,1.0,-0.082495,-0.054489,-0.159794
2,JILL,2018-04-16 16:51:37,2018-04-12,Dir,499799.0,1.000000,0.0,4.0,False,1.0,1.0,-0.050752,0.077068,0.578947
3,OPK,2018-04-17 06:53:38,2018-04-16,CEO,150950.0,0.000263,0.0,1.0,True,1.0,1.0,-0.035370,0.569132,0.945338
4,HCAP,2018-04-10 17:47:08,2018-04-06,10%,72349.0,0.007462,1.0,4.0,False,1.0,1.0,0.006232,-0.028599,0.070170


## 2. Feature Engineering

### 2.1 Insider ID

In [3]:
# Crear ID único para cada insider
df['insider_id'] = df['ticker'] + '_' + df['insider_role']

print(f"✅ insider_id creado")
print(f"   Insiders únicos: {df['insider_id'].nunique():,}")

✅ insider_id creado
   Insiders únicos: 12,399


### 2.2 Track Record del Insider (CRÍTICO)

In [4]:
# Ordenar cronológicamente
df = df.sort_values(['insider_id', 'filing_date']).reset_index(drop=True)

# Número de transacciones previas (experiencia del insider)
df['insider_trade_count'] = df.groupby('insider_id').cumcount()

# Días desde última transacción (frecuencia de trading)
df['days_since_last_trade'] = df.groupby('insider_id')['trade_date'].diff().dt.days

print("✅ Track record calculado")
print(f"\n📊 Estadísticas:")
print(df[['insider_trade_count', 'days_since_last_trade']].describe())

✅ Track record calculado

📊 Estadísticas:
       insider_trade_count  days_since_last_trade
count         45201.000000           32802.000000
mean              9.246079             153.953844
std              24.540287             315.408742
min               0.000000            -187.000000
25%               0.000000               2.000000
50%               2.000000              13.000000
75%               7.000000             160.000000
max             340.000000            2916.000000


### 2.3 Timing Features

In [5]:
# Día de la semana (0=Lunes, 4=Viernes)
df['trade_day_of_week'] = df['trade_date'].dt.dayofweek

print("✅ Timing features creados")
print(f"\n📊 Distribución por día:")
print(df['trade_day_of_week'].value_counts().sort_index())

✅ Timing features creados

📊 Distribución por día:
trade_day_of_week
0.0    9138
1.0    9287
2.0    8298
3.0    8820
4.0    9614
5.0      16
6.0      28
Name: count, dtype: int64


### 2.4 Calidad del Cluster

In [6]:
# % de C-levels en el cluster (señal de calidad)
df['cluster_c_level_pct'] = df.groupby(['ticker', 'trade_date'])['is_c_level'].transform('mean')

print("✅ Calidad del cluster calculada")
print(f"\n📊 Estadísticas:")
print(df['cluster_c_level_pct'].describe())

✅ Calidad del cluster calculada

📊 Estadísticas:
count     45201.0
unique       29.0
top           0.0
freq      34830.0
Name: cluster_c_level_pct, dtype: float64


## 3. Limpieza de Redundancias

In [7]:
# Analizar correlación entre owned_pct_change y delta_owned
if 'owned_pct_change' in df.columns:
    corr = df[['owned_pct_change', 'delta_owned']].corr()
    print("📊 Correlación:")
    print(corr)
    
    corr_value = corr.iloc[0, 1]
    if abs(corr_value) > 0.6:  # Umbral para correlación moderada-alta
        print(f"\n⚠️  Correlación moderada-alta ({corr_value:.3f})")
        print(f"   Eliminando 'owned_pct_change' (menos interpretable que delta_owned)")
        df = df.drop(columns=['owned_pct_change'])
        print(f"   ✅ 'owned_pct_change' eliminado")
    else:
        print(f"\n✅ Correlación baja ({corr_value:.3f})")

📊 Correlación:
                  owned_pct_change  delta_owned
owned_pct_change          1.000000    -0.009876
delta_owned              -0.009876     1.000000

✅ Correlación baja (-0.010)


## 4. Guardar Dataset Final

In [8]:
output_path = project_root / "data" / "processed" / "insider_trades_final.parquet"
df.to_parquet(output_path, index=False)

print(f"\n✅ Dataset final guardado: {output_path}")
print(f"\n📊 Resumen final:")
print(f"  Registros: {len(df):,}")
print(f"  Columnas: {len(df.columns)}")
print(f"  Insiders únicos: {df['insider_id'].nunique():,}")

print(f"\n📋 Columnas finales:")
for col in df.columns:
    non_null = df[col].notna().sum()
    pct = 100 * non_null / len(df)
    dtype = df[col].dtype
    print(f"  {col:30s} ({str(dtype):15s}) - {pct:5.1f}% completo")


✅ Dataset final guardado: ../../data/processed/insider_trades_final.parquet

📊 Resumen final:
  Registros: 45,242
  Columnas: 19
  Insiders únicos: 12,399

📋 Columnas finales:
  ticker                         (object         ) -  99.9% completo
  filing_date                    (datetime64[ns] ) -  99.9% completo
  trade_date                     (datetime64[ns] ) -  99.9% completo
  insider_role                   (object         ) -  99.9% completo
  transaction_value              (float64        ) -  99.9% completo
  delta_owned                    (float64        ) -  99.9% completo
  owned_pct_change               (float64        ) -  99.9% completo
  reporting_lag                  (float64        ) -  99.9% completo
  is_c_level                     (object         ) -  99.9% completo
  cluster_buy                    (float64        ) -  99.9% completo
  cluster_size                   (float64        ) -  99.9% completo
  return_1w                      (float64        ) -  90.0% comp

## 5. Resumen del Dataset Final

### Features para el Modelo ML:

**Identificadores:**
- `ticker`: Símbolo
- `filing_date`: Fecha de reporte
- `trade_date`: Fecha de transacción  
- `insider_id`: ID del insider

**Features del Insider:**
- `insider_role`: Rol (CEO, CFO, Dir, etc.)
- `transaction_value`: Valor en USD
- `delta_owned`: % cambio en holdings
- `is_c_level`: Boolean C-level
- `insider_trade_count`: Experiencia (# ops previas) ⭐
- `days_since_last_trade`: Frecuencia de trading ⭐

**Features del Evento:**
- `reporting_lag`: Días entre trade y filing
- `cluster_buy`: # insiders mismo día
- `cluster_size`: # insiders en 7 días
- `cluster_c_level_pct`: % C-levels en cluster ⭐
- `trade_day_of_week`: Día de la semana ⭐

**Target Variables:**
- `return_1w`: Retorno 1 semana
- `return_1m`: Retorno 1 mes
- `return_3m`: Retorno 3 meses

**Total:** 18 columnas (4 IDs + 10 features + 3 targets)